In [ ]:
from transformers import RagRetriever, RagSequenceForGeneration, RagTokenizer

In [ ]:
!pip install faiss-cpu# to avoid faiss libarray installation error(seprata for CPU and GPU)

Hugging Face datasets version 4.0.0 and above completely dropped support for Python loading scripts (like wiki_dpr.py) due to security and maintenance concerns.

Most official Hugging Face datasets have automatically generated Parquet mirrors. **You can bypass the legacy loading script by targeting the converted Parquet branch directly via the revision argument(done in next block)**.

In [ ]:
from datasets import load_dataset


# Point directly to the auto-converted Parquet files
dataset = load_dataset(
    "facebook/wiki_dpr", # to avert "This error occurs because newer versions of huggingface_hub (v1.16+)
    #require a namespace/name structure for repository IDs instead of bare single-segment names". Repository id must be 'namespace/name', got 'wiki_dpr'.
    "psgs_w100.nq.exact",  # or whichever configuration you need
    trust_remote_code=True, #to avert "ValueError: BuilderConfig 'psgs_w100.nq.exact' not found. Available: ['default']"
    #So Pass the repository name explicitly as the first argument
    revision="refs/convert/parquet"
)

The below code will run. But takes time and occupies much of RAM, thereby increasing notebook size. Better to comment and call datasets facebook/wiki in retriever by aleterd code(2nd block afer this)

In [ ]:
'''
dataset = load_dataset(
    "facebook/wiki_dpr",
    "default",  # or whichever configuration you need
    revision="refs/convert/parquet"
)
'''

Initialializimg tokenizer and retriever.

In [ ]:
!pip install --upgrade datasets pyarrow fsspec

In [ ]:
!rm -rf /root/.cache/huggingface/datasets/# to free up disc
!rm -rf /root/.cache/huggingface/hub/

In [ ]:
rm -rf /root/.cache/huggingface/hub/datasets--facebook--wiki_dpr

In [ ]:
!rm -rf /content/your_folder/

In [ ]:
!du -h --max-depth=1 /content

148K	/content/.config
55M	/content/sample_data
55M	/content


In [ ]:
from datasets import load_dataset, Features, Value, Sequence
tokenizer = RagTokenizer.from_pretrained("facebook/rag-sequence-nq")

'''
retriever = RagRetriever.from_pretrained(
    "facebook/rag-sequence-nq", dataset="facebook/wiki_dpr", index_name="compressed"
)
'''

'''
retriever = RagRetriever.from_pretrained(
    "facebook/rag-sequence-nq",
    index_name="exact",
    use_dummy_dataset=False,
    dataset="facebook/wiki_dpr"  # Ensure this points to the full path
)#refer 2nd last block comments
'''


# Define the actual schema present in the Parquet file
actual_features = Features({
    "id": Value("string"),
    "text": Value("string"),
    "title": Value("string"),
    "embeddings": Sequence(Value("float32"))  # Matches the missing column
})


'''
retriever = RagRetriever.from_pretrained(
    "facebook/rag-sequence-nq",
    index_name="compressed",
    use_dummy_dataset=False,

    dataset= load_dataset(
    "facebook/wiki_dpr",
    "default",  # or whichever configuration you need
    revision="refs/convert/parquet",
    download_mode="force_redownload"# to tackle "DatasetGenerationError: An error occurred while generating the dataset"(Failed to
    #read file '/root/.cache/huggingface/hub/datasets--facebook--)

) # Ensure this points to the full path
)
'''


retriever = RagRetriever.from_pretrained(
    "facebook/rag-sequence-nq",
    index_name="compressed",
    use_dummy_dataset=False,

    dataset= load_dataset(
    "facebook/wiki_dpr",
    "default",  # or whichever configuration you need
    revision="refs/convert/parquet",
    features=actual_features

) # Ensure this points to the full path
)


# Issue to to disc overflow in running above block, now approach 2

In [ ]:
!pip install faiss-cpu sentence-transformers transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 15.5 MB/s eta 0:00:00


**1.)Initialize the Knowledge Base and Embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Sample external documents (knowledge base)
documents = [
    "The capital of France is Paris, known for the Eiffel Tower.",
    "The currency of Japan is the Japanese Yen.",
    "Retrieval-Augmented Generation (RAG) combines external data with large language models.",
    "RAGs use of retrieval outputs giving most probable similar chunks to the query question from knowlege base",
    "The output chunks appeneded with query is feeded in generator LLM which uses its pretrained/fine-tunefd knowledge(depending on query instruction), to give refined o/p",
    "Mount Everest is the highest mountain peak in the world above sea level."
]

# Load a pre-trained embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the documents
doc_embeddings = embed_model.encode(documents, convert_to_numpy=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**2. Set Up the FAISS Retriever**

Index the document embeddings using FAISS for fast similarity search.

In [ ]:
# Initialize FAISS index based on vector dimension
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add document vectors to the index(analogous to custom DB)
index.add(doc_embeddings)

**3. Define the Retrieval and Generation Logic**

Query the FAISS index to find the most relevant passage, then pass it along with the user's prompt to a pre-trained text generation model.


In [ ]:
from transformers import pipeline

# Load a lightweight pre-trained generator model
generator = pipeline("text-generation", model="google/flan-t5-small")

def rag_pipeline(query: str, top_k: int = 1):
    # 1. Embed the query and retrieve top-k matching documents
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    for elm in zip(distances, indices):
      print(f"Distance is:-{elm[0]}")
      print(f"Indices are:-{elm[1]}")

    print("@"*100)

    retrieved_context = documents[indices[0][0]]

    # 2. Construct the grounded prompt
    prompt = f"Context: {retrieved_context}\n\nQuestion: {query}\n\nAnswer:"

    # 3. Generate response
    response = generator(prompt, max_length=50, do_sample=False)
    print(response)
    print("~"*50)
    return response[0]['generated_text'], retrieved_context, prompt


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

In [ ]:
# Run a query through the pipeline
user_query = "What is RAG? Explain their working mechanism?"
user_query ="What is RAG?"
answer, source, prompt = rag_pipeline(user_query)

print(f"Query: {user_query}")
print("%"*50)
print(f"Prompt: {prompt}")
print("!"*50)
print(f"Retrieved Source: {source}")
print("#"*50)
print(f"Generated Answer: {answer}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Distance is:-[1.4003041]
Indices are:-[2]
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
[{'generated_text': 'Context: Retrieval-Augmented Generation (RAG) combines external data with large language models.\n\nQuestion: What is RAG?\n\nAnswer:'}]
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Query: What is RAG?
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
Prompt: Context: Retrieval-Augmented Generation (RAG) combines external data with large language models.

Question: What is RAG?

Answer:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Retrieved Source: Retrieval-Augmented Generation (RAG) combines external data with large language models.
##################################################
Generated Answer: Context: Retrieval-Augmented Generation (RAG) combines external data with large language models.

Question: What is RAG?

Answer:


# RAG 2

To run the models, we will use ollama, a command line tool that allows you to run models from Hugging Face. **With ollama, you don't need to have access to a server or cloud service to run the models. You can run the models directly on your computer.**

Blocks to ensure ollama is running

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
!sudo apt update && sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Ign:11 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:11 http://security.ubuntu.com/ubuntu jammy-security InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
171 packages can be upgraded. Run 'apt list --upgradable' to see them.

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &

In [ ]:
# Verify the background service is active
!curl http://127.0.0.1:11434

# Download your chosen model
!ollama pull llama3

Ollama is running


In [ ]:
!pip install ollama

In [ ]:
!pkill ollama# to kill active session
!ollama serve &

time=2026-08-04T15:28:53.229Z level=INFO source=routes.go:1947 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

**open a terminal and run the following command to download the required models**

In [ ]:
# Enable GPU support, turn off unneeded features, and use all  CPU cores(-1) available on free Colab(else model download in ollama model would take a lot of time)
%env CMAKE_ARGS=-DGGML_CUDA=on -DLLAVA_BUILD=off
%env CMAKE_BUILD_PARALLEL_LEVEL=-1

!ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
!ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
#!ollama run hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest

since "!ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF" taking long time to download, we'll download it differently later

**Requesting the dataset & Loading the dataset**

In [ ]:
import requests

# 1. Get the raw text file from Hugging Face
url = "https://huggingface.co"
response = requests.get(url)
response.encoding = "utf-8"

# 2. Save it locally in your current directory (e.g., /content/)
with open("cat-facts.txt", "w", encoding="utf-8") as f:
  f.write(response.text)

In [ ]:
# 3. Now read the local file safely

with open('cat-facts.txt', 'r', encoding="utf-8") as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')

Loaded 104 entries


**Implement vector database**

In [ ]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
#LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

Added chunk 1/104 to the database
Added chunk 2/104 to the database
Added chunk 3/104 to the database
Added chunk 4/104 to the database
Added chunk 5/104 to the database
Added chunk 6/104 to the database
Added chunk 7/104 to the database
Added chunk 8/104 to the database
Added chunk 9/104 to the database
Added chunk 10/104 to the database
Added chunk 11/104 to the database
Added chunk 12/104 to the database
Added chunk 13/104 to the database
Added chunk 14/104 to the database
Added chunk 15/104 to the database
Added chunk 16/104 to the database
Added chunk 17/104 to the database
Added chunk 18/104 to the database
Added chunk 19/104 to the database
Added chunk 20/104 to the database
Added chunk 21/104 to the database
Added chunk 22/104 to the database
Added chunk 23/104 to the database
Added chunk 24/104 to the database
Added chunk 25/104 to the database
Added chunk 26/104 to the database
Added chunk 27/104 to the database
Added chunk 28/104 to the database
Added chunk 29/104 to the dat

**Define cosine similarity and retreival fn**

In [ ]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]

**Generation phrase**:-

1)The retreival fn above will print topk(=3) chunks most similar to our query.

2)These top chunks will be added into the prompt and the chatbot shall generate reponse based on retrieved knowledge.

In [ ]:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

Ask me a question: What are peculiar behavioural traits of cats?
Retrieved knowledge:
 - (similarity: 0.43) 				customProperties: {

 - (similarity: 0.43) 		<title>Hugging Face – The AI community building the future.</title>

 - (similarity: 0.42) 		<!-- Stripe -->



**Output1:- genreation via ollama.chat method**

In [ ]:
#use the ollama to generate the response. In this example, we will use instruction_prompt as system message
stream = ollama.chat(
  #model =llm,
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Chatbot response:
The stereotypical feline persona! Cats are known for their curious, independent, and sometimes quirky nature. Here are some peculiar behavioral traits associated with cats:

1. **Grooming as self-care**: Cats spend a significant amount of time grooming themselves, which is thought to be an evolutionary leftover from their wild ancestors.
2. **Sleep patterns**: Cats are notorious for their love of sleep. They can spend up to 16 hours a day snoozing!
3. **Claws out**: When cats feel threatened or scared, they may extend their claws as a defensive mechanism. However, this behavior is usually a sign that the cat feels safe and relaxed.
4. **Scratching furniture (and sometimes humans)**: Cats have sharp claws that they use to scratch and mark their territory. This behavior can be annoying for owners who prefer a low-maintenance pet.
5. **Fur-throwing**: When cats feel threatened or scared, they may release a strong, unpleasant odor from their anal glands, which is why you 

**Output2**:- through install llama_cpp library via pip and **load model :- "bartowski/Llama-3.2-1B-Instruct-GGUF"** in python(without ollama framework).

This takes more time

load "ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF" differently. we'll bypass it for huggingface and directly load it(Use llama-cpp-python or standard llama.cpp directly, which loads GGUF files much faster )

In [ ]:
#!pip install --upgrade pip setuptools wheel cmake

Run this command for a 10-second(with **extra index url**) installation with/without GPU support

In [ ]:
# Enable GPU support, turn off unneeded features, and use all  CPU cores(-1) available on free Colab
%env CMAKE_ARGS=-DGGML_CUDA=on -DLLAVA_BUILD=off
%env CMAKE_BUILD_PARALLEL_LEVEL=-1

#!pip install llama-cpp-python --no-cache-dir --force-reinstall --upgrade
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/gpu

env: CMAKE_ARGS=-DGGML_CUDA=on -DLLAVA_BUILD=off
env: CMAKE_BUILD_PARALLEL_LEVEL=-1
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/gpu


In [ ]:
from llama_cpp import Llama
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_M.gguf"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           gen

In [ ]:
# 5. Format using Chat Completion (Ollama/OpenAI compatible format)
messages = [
    {"role": "system", "content": instruction_prompt},
    {"role": "user", "content": input_query}
]

# 6. Generate the response
print("Generating response...")
response = llm.create_chat_completion(
    messages=messages,
    #temperature=0.1,
    #max_tokens=256
)

# 7. Print the result
output_text = response["choices"][0]["message"]["content"]
print("\n=== AI Response ===")
print(output_text)

Generating response...


Llama.generate: 53 prefix-match hit, remaining 47 prompt tokens to eval
llama_perf_context_print:        load time =    5935.06 ms
llama_perf_context_print: prompt eval time =    2518.17 ms /    47 tokens (   53.58 ms per token,    18.66 tokens per second)
llama_perf_context_print:        eval time =   51031.00 ms /   411 runs   (  124.16 ms per token,     8.05 tokens per second)
llama_perf_context_print:       total time =   53983.63 ms /   458 tokens
llama_perf_context_print:    graphs reused =        409



=== AI Response ===
Cats are known for their unique and sometimes quirky behavioral traits. Here are some peculiar ones:

1. **Grooming habits**: Cats spend a significant amount of time grooming themselves, which can be a sign of stress or anxiety. They may also groom each other as a form of social bonding.

2. **Hunting instinct**: Even domesticated cats have a strong prey drive and may exhibit hunting behavior, such as stalking and pouncing on toys or small animals.

3. **Territorial marking**: Cats have scent glands in their urine and feces, and they may mark their territory by spraying or scratching surfaces.

4. **Sleep patterns**: Cats are notorious for their love of sleep, but they also have a unique sleep pattern. They can spend up to 16 hours a day sleeping, and they often take multiple naps throughout the day.

5. **Affection on their terms**: Cats are not always interested in cuddling or playing, and they may show affection on their own terms. This can be confusing for owne